
# Traffic Demand Prediction — Advanced High-Score Pipeline

This notebook implements a strong competition-grade pipeline for traffic demand prediction with:

- Advanced temporal features
- Geohash hierarchy features
- Fold-based target encoding
- Spatial statistics
- Interaction features
- CatBoost tuned model
- LightGBM ensemble
- Log-transformed target
- Leakage-safe validation
- Weighted ensemble submission

Expected improvements:
- Better leaderboard generalization
- Reduced leakage
- Better geo-temporal learning


In [39]:

# =========================
# INSTALLS (if needed)
# =========================

# Uncomment if required

# !pip install catboost lightgbm pygeohash category_encoders


In [40]:

# =========================
# IMPORTS
# =========================

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

import pygeohash as pgh


In [41]:

# =========================
# LOAD DATA
# =========================

train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')
sample = pd.read_csv('dataset/sample_submission.csv')

print(train.shape)
print(test.shape)

train.head()


(77299, 11)
(41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [42]:

# =========================
# BASIC CLEANING
# =========================

combined = pd.concat([
    train.drop(columns=['demand']),
    test
], axis=0).reset_index(drop=True)

target = train['demand'].copy()

print(combined.shape)


(119077, 10)


In [43]:

# =========================
# GEOHASH DECODING
# =========================

def decode_geohash(gh):
    try:
        lat, lon = pgh.decode(gh)
        return lat, lon
    except:
        return np.nan, np.nan

combined[['latitude', 'longitude']] = combined['geohash'].apply(
    lambda x: pd.Series(decode_geohash(x))
)

combined.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,latitude,longitude
0,0,qp02z1,48,0:0,NaN,1,Not Allowed,No,NaN,NaN,-5.484924,90.664673
1,1,qp02zt,48,0:0,Residential,3,Allowed,Yes,31.104565,Sunny,-5.462952,90.686646
2,2,qp08bj,48,0:0,Residential,1,Not Allowed,No,25.919267,Sunny,-5.462952,90.708618
3,3,qp08gt,48,0:0,Residential,1,Not Allowed,No,NaN,Rainy,-5.462952,90.862427
4,4,qp02zq,48,0:0,Residential,1,Not Allowed,No,10.803667,Rainy,-5.457458,90.675659


In [44]:

# =========================
# GEOHASH HIERARCHY FEATURES
# =========================

combined['geo_4'] = combined['geohash'].astype(str).str[:4]
combined['geo_5'] = combined['geohash'].astype(str).str[:5]
combined['geo_6'] = combined['geohash'].astype(str).str[:6]

combined[['geo_4','geo_5','geo_6']].head()


,geo_4,geo_5,geo_6
0,qp02,qp02z,qp02z1
1,qp02,qp02z,qp02zt
2,qp08,qp08b,qp08bj
3,qp08,qp08g,qp08gt
4,qp02,qp02z,qp02zq


In [45]:

# =========================
# TIME FEATURES
# =========================

combined['timestamp'] = pd.to_datetime(combined['timestamp'])

combined['hour'] = combined['timestamp'].dt.hour
combined['minute'] = combined['timestamp'].dt.minute
combined['dayofweek'] = combined['timestamp'].dt.dayofweek
combined['weekofyear'] = combined['timestamp'].dt.isocalendar().week.astype(int)

combined['minutes'] = combined['hour'] * 60 + combined['minute']

# Cyclic encoding
combined['hour_sin'] = np.sin(2*np.pi*combined['hour']/24)
combined['hour_cos'] = np.cos(2*np.pi*combined['hour']/24)

combined['minute_sin'] = np.sin(2*np.pi*combined['minutes']/1440)
combined['minute_cos'] = np.cos(2*np.pi*combined['minutes']/1440)

# Peak traffic indicators
combined['is_peak_morning'] = combined['hour'].between(7,10).astype(int)
combined['is_peak_evening'] = combined['hour'].between(16,20).astype(int)
combined['is_night'] = combined['hour'].between(0,5).astype(int)

combined['is_weekend'] = combined['dayofweek'].isin([5,6]).astype(int)

combined.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,weekofyear,minutes,hour_sin,hour_cos,minute_sin,minute_cos,is_peak_morning,is_peak_evening,is_night,is_weekend
0,0,qp02z1,48,1-01-01,NaN,1,Not Allowed,No,NaN,NaN,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
1,1,qp02zt,48,1-01-01,Residential,3,Allowed,Yes,31.104565,Sunny,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
2,2,qp08bj,48,1-01-01,Residential,1,Not Allowed,No,25.919267,Sunny,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
3,3,qp08gt,48,1-01-01,Residential,1,Not Allowed,No,NaN,Rainy,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0
4,4,qp02zq,48,1-01-01,Residential,1,Not Allowed,No,10.803667,Rainy,...,1,0,0.0,1.0,0.0,1.0,0,0,1,0


In [46]:
# =========================
# BINARY / BOOLEAN ENCODING
# =========================

binary_cols = [
    'LargeVehicles',
    'Landmarks'
]

for col in binary_cols:

    combined[col] = combined[col].astype(str).str.lower()

    combined[col] = combined[col].map({
        'yes': 1,
        'no': 0,
        'true': 1,
        'false': 0,
        '1': 1,
        '0': 0
    })

    combined[col] = combined[col].fillna(0)

print(combined[binary_cols].head())

   LargeVehicles  Landmarks
0            0.0          0
1            0.0          1
2            0.0          0
3            0.0          0
4            0.0          0


In [47]:

# =========================
# INTERACTION FEATURES
# =========================

combined['temp_hour'] = combined['Temperature'] * combined['hour']

combined['lanes_temp'] = (
    combined['NumberofLanes'] * combined['Temperature']
)

combined['vehicle_lane'] = (
    combined['LargeVehicles'] * combined['NumberofLanes']
)

combined['road_capacity'] = (
    combined['NumberofLanes'] * (combined['LargeVehicles'] + 1)
)

combined['road_weather'] = (
    combined['RoadType'].astype(str) + '_' +
    combined['Weather'].astype(str)
)

combined.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,minute_cos,is_peak_morning,is_peak_evening,is_night,is_weekend,temp_hour,lanes_temp,vehicle_lane,road_capacity,road_weather
0,0,qp02z1,48,1-01-01,NaN,1,0.0,0,NaN,NaN,...,1.0,0,0,1,0,NaN,NaN,0.0,1.0,NaN
1,1,qp02zt,48,1-01-01,Residential,3,0.0,1,31.104565,Sunny,...,1.0,0,0,1,0,0.0,93.313694,0.0,3.0,Residential_Sunny
2,2,qp08bj,48,1-01-01,Residential,1,0.0,0,25.919267,Sunny,...,1.0,0,0,1,0,0.0,25.919267,0.0,1.0,Residential_Sunny
3,3,qp08gt,48,1-01-01,Residential,1,0.0,0,NaN,Rainy,...,1.0,0,0,1,0,NaN,NaN,0.0,1.0,Residential_Rainy
4,4,qp02zq,48,1-01-01,Residential,1,0.0,0,10.803667,Rainy,...,1.0,0,0,1,0,0.0,10.803667,0.0,1.0,Residential_Rainy


In [48]:

# =========================
# MISSING VALUE FEATURES
# =========================

for col in combined.columns:
    if combined[col].isnull().sum() > 0:
        combined[f'{col}_missing'] = combined[col].isnull().astype(int)
# =========================
# STRICT MISSING VALUE HANDLING
# =========================

# Explicit numeric columns
numeric_columns = [
    'day',
    'NumberofLanes',
    'Temperature',
    'latitude',
    'longitude',
    'minutes',
    'hour',
    'minute',
    'dayofweek',
    'weekofyear',
    'hour_sin',
    'hour_cos',
    'minute_sin',
    'minute_cos',
    'is_peak_morning',
    'is_peak_evening',
    'is_night',
    'is_weekend',
    'temp_hour',
    'lanes_temp',
    'vehicle_lane',
    'road_capacity',
    'geo_density',
    'geo_hour_mean'
]

# Add ALL target encoding cols automatically
numeric_columns += [
    col for col in combined.columns
    if '_te' in col
]

# Convert numeric cols safely
for col in numeric_columns:

    if col in combined.columns:

        combined[col] = pd.to_numeric(
            combined[col],
            errors='coerce'
        )

        combined[col] = combined[col].fillna(
            combined[col].median()
        )

# Handle categorical columns separately
for col in combined.columns:

    if col not in numeric_columns:

        combined[col] = combined[col].astype(str)

        combined[col] = combined[col].fillna('Unknown')

print("Missing value handling complete.")

Missing value handling complete.


In [49]:

# =========================
# OPTIONAL SPATIAL CLUSTERING
# =========================

kmeans = KMeans(n_clusters=20, random_state=42)

combined['spatial_cluster'] = kmeans.fit_predict(
    combined[['latitude','longitude']]
)

combined['spatial_cluster'].value_counts().head()


spatial_cluster
7     9238
6     7142
3     7006
12    6885
15    6753
Name: count, dtype: int64

In [50]:

# =========================
# SPLIT BACK
# =========================

train_df = combined.iloc[:len(train)].copy()
test_df = combined.iloc[len(train):].copy()

train_df['demand'] = target.values

print(train_df.shape)
print(test_df.shape)


(77299, 41)
(41778, 40)


In [51]:

# =========================
# LOG TRANSFORM TARGET
# =========================

train_df['target_log'] = np.log1p(train_df['demand'])

train_df[['demand','target_log']].head()


,demand,target_log
0,0.048804,0.047651
1,0.118507,0.111995
2,0.027132,0.026770
3,0.003272,0.003266
4,0.010819,0.010761


In [52]:

# =========================
# LEAKAGE SAFE TARGET ENCODING
# =========================

def target_encode_oof(train, test, column, target, n_splits=5):

    train_encoded = np.zeros(len(train))
    test_encoded = np.zeros(len(test))

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(train):

        X_train = train.iloc[train_idx]
        X_val = train.iloc[val_idx]

        means = X_train.groupby(column)[target].mean()

        train_encoded[val_idx] = X_val[column].map(means)

    global_mean = train[target].mean()

    train_encoded = np.where(
        np.isnan(train_encoded),
        global_mean,
        train_encoded
    )

    final_means = train.groupby(column)[target].mean()

    test_encoded = test[column].map(final_means)
    test_encoded = test_encoded.fillna(global_mean)

    return train_encoded, test_encoded


In [53]:

# =========================
# APPLY TARGET ENCODING
# =========================

target_cols = [
    'geohash',
    'geo_4',
    'geo_5',
    'geo_6',
    'RoadType',
    'Weather',
    'road_weather'
]

for col in target_cols:

    train_df[f'{col}_te'], test_df[f'{col}_te'] = target_encode_oof(
        train_df,
        test_df,
        col,
        'target_log'
    )

print("Target encoding completed.")


Target encoding completed.


In [54]:

# =========================
# GEO STATISTICS FEATURES
# =========================

geo_density = train_df.groupby('geohash').size()

train_df['geo_density'] = train_df['geohash'].map(geo_density)
test_df['geo_density'] = test_df['geohash'].map(geo_density)

geo_hour_mean = train_df.groupby(['geohash','hour'])['target_log'].mean()

train_df['geo_hour_mean'] = train_df.set_index(
    ['geohash','hour']
).index.map(geo_hour_mean)

test_df['geo_hour_mean'] = test_df.set_index(
    ['geohash','hour']
).index.map(geo_hour_mean)

global_mean = train_df['target_log'].mean()

train_df['geo_hour_mean'] = train_df['geo_hour_mean'].fillna(global_mean)
test_df['geo_hour_mean'] = test_df['geo_hour_mean'].fillna(global_mean)

train_df.head()


,Index,geohash,day,timestamp,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,...,target_log,geohash_te,geo_4_te,geo_5_te,geo_6_te,RoadType_te,Weather_te,road_weather_te,geo_density,geo_hour_mean
0,0,qp02z1,48,1-01-01 00:00:00,Unknown,1,0.0,0,16.415162,Unknown,...,0.047651,0.034941,0.124132,0.127130,0.034941,0.087700,0.086759,0.086200,33,0.034137
1,1,qp02zt,48,1-01-01 00:00:00,Residential,3,0.0,1,31.104565,Sunny,...,0.111995,0.178883,0.120919,0.123671,0.178883,0.054583,0.083042,0.054190,89,0.131178
2,2,qp08bj,48,1-01-01 00:00:00,Residential,1,0.0,0,25.919267,Sunny,...,0.026770,0.117081,0.053219,0.070515,0.117081,0.054382,0.083218,0.054132,67,0.064492
3,3,qp08gt,48,1-01-01 00:00:00,Residential,1,0.0,0,16.415162,Rainy,...,0.003266,0.014621,0.053491,0.032445,0.014621,0.054583,0.083695,0.055057,42,0.011124
4,4,qp02zq,48,1-01-01 00:00:00,Residential,1,0.0,0,10.803667,Rainy,...,0.010761,0.029818,0.124132,0.127130,0.029818,0.054516,0.083162,0.054814,36,0.020447


In [55]:

# =========================
# FEATURE LIST
# =========================

drop_cols = [
    'Index',
    'timestamp',
    'geohash',
    'demand',
    'target_log'
]

features = [
    col for col in train_df.columns
    if col not in drop_cols
]

categorical_features = [
    col for col in features
    if train_df[col].dtype == 'object'
]

print("Total Features:", len(features))
print("Categorical:", len(categorical_features))


Total Features: 46
Categorical: 0


In [56]:
train_df['day'] = pd.to_numeric(
    train_df['day'],
    errors='coerce'
)

test_df['day'] = pd.to_numeric(
    test_df['day'],
    errors='coerce'
)

In [57]:
# =========================
# SAFE VALIDATION SPLIT
# =========================

unique_days = sorted(train_df['day'].dropna().unique())

print("Unique Days:", unique_days)

# Use last 20% days for validation
split_idx = int(len(unique_days) * 0.8)

train_days = unique_days[:split_idx]
valid_days = unique_days[split_idx:]

print("Train Days:", train_days)
print("Valid Days:", valid_days)

train_mask = train_df['day'].isin(train_days)
val_mask = train_df['day'].isin(valid_days)

X_train = train_df.loc[train_mask, features]
y_train = train_df.loc[train_mask, 'target_log']

X_valid = train_df.loc[val_mask, features]
y_valid = train_df.loc[val_mask, 'target_log']

X_test = test_df[features]

print(X_train.shape)
print(X_valid.shape)

assert len(X_train) > 0
assert len(X_valid) > 0
assert len(y_train) > 0
assert len(y_valid) > 0


Unique Days: [np.int64(48), np.int64(49)]
Train Days: [np.int64(48)]
Valid Days: [np.int64(49)]
(69427, 46)
(7872, 46)


In [58]:
for col in X_train.columns:

    if X_train[col].dtype == 'object':

        print(col, "OBJECT")

    if X_train[col].astype(str).str.contains('Unknown').any():

        print(col, "contains Unknown")

RoadType contains Unknown
Weather contains Unknown
road_weather contains Unknown


In [60]:
# =========================
# FORCE CATEGORICAL TYPES
# =========================

categorical_features = [
    'geohash',
    'geo_4',
    'geo_5',
    'geo_6',
    'RoadType',
    'Weather',
    'road_weather'
]

# Keep only existing cols
categorical_features = [
    col for col in categorical_features
    if col in X_train.columns
]

# Convert categoricals to string
for col in categorical_features:

    X_train[col] = X_train[col].astype(str)
    X_valid[col] = X_valid[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# IMPORTANT:
# CatBoost prefers column INDEXES
cat_feature_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

print(cat_feature_indices)
print(categorical_features)

[9, 10, 11, 1, 6, 29]
['geo_4', 'geo_5', 'geo_6', 'RoadType', 'Weather', 'road_weather']


In [61]:

# =========================
# CATBOOST MODEL
# =========================

cat_model = CatBoostRegressor(
    iterations=12000,
    learning_rate=0.01,
    depth=8,
    loss_function='RMSE',
    eval_metric='RMSE',
    random_seed=42,
    l2_leaf_reg=5,
    bagging_temperature=1,
    random_strength=1,
    bootstrap_type='Bayesian',
    boosting_type='Ordered',
    verbose=500
)

cat_model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    cat_features=cat_feature_indices,
    use_best_model=True,
    early_stopping_rounds=500
)

cat_valid = cat_model.predict(X_valid)
cat_test = cat_model.predict(X_test)


0:	learn: 0.1074396	test: 0.1120963	best: 0.1120963 (0)	total: 172ms	remaining: 34m 25s
500:	learn: 0.0185423	test: 0.0466471	best: 0.0463255 (321)	total: 40.6s	remaining: 15m 32s
Stopped by overfitting detector  (500 iterations wait)

bestTest = 0.04632551122
bestIteration = 321

Shrink model to first 322 iterations.


In [63]:
# =========================
# CLEAN TYPES FOR LIGHTGBM
# =========================

# Missing indicator columns should be int
missing_cols = [
    col for col in X_train.columns
    if '_missing' in col
]

for col in missing_cols:

    X_train[col] = pd.to_numeric(
        X_train[col],
        errors='coerce'
    ).fillna(0).astype(int)

    X_valid[col] = pd.to_numeric(
        X_valid[col],
        errors='coerce'
    ).fillna(0).astype(int)

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors='coerce'
    ).fillna(0).astype(int)

# Binary columns
binary_cols = [
    'LargeVehicles',
    'Landmarks'
]

for col in binary_cols:

    X_train[col] = pd.to_numeric(
        X_train[col],
        errors='coerce'
    ).fillna(0).astype(int)

    X_valid[col] = pd.to_numeric(
        X_valid[col],
        errors='coerce'
    ).fillna(0).astype(int)

    X_test[col] = pd.to_numeric(
        X_test[col],
        errors='coerce'
    ).fillna(0).astype(int)

# Actual categorical columns
categorical_features = [
    'geohash',
    'geo_4',
    'geo_5',
    'geo_6',
    'RoadType',
    'Weather',
    'road_weather'
]

categorical_features = [
    col for col in categorical_features
    if col in X_train.columns
]

# Convert ONLY real categoricals
for col in categorical_features:

    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

print(X_train.dtypes)

day                        int64
RoadType                category
NumberofLanes              int64
LargeVehicles              int64
Landmarks                  int64
Temperature              float64
Weather                 category
latitude                 float64
longitude                float64
geo_4                   category
geo_5                   category
geo_6                   category
hour                       int32
minute                     int32
dayofweek                  int32
weekofyear                 int64
minutes                    int32
hour_sin                 float64
hour_cos                 float64
minute_sin               float64
minute_cos               float64
is_peak_morning            int64
is_peak_evening            int64
is_night                   int64
is_weekend                 int64
temp_hour                float64
lanes_temp               float64
vehicle_lane             float64
road_capacity            float64
road_weather            category
RoadType_m

In [64]:

# =========================
# LIGHTGBM MODEL
# =========================

for col in categorical_features:
    X_train[col] = X_train[col].astype('category')
    X_valid[col] = X_valid[col].astype('category')
    X_test[col] = X_test[col].astype('category')

lgb_model = LGBMRegressor(
    n_estimators=6000,
    learning_rate=0.01,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)]
)

lgb_valid = lgb_model.predict(X_valid)
lgb_test = lgb_model.predict(X_test)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008020 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3610
[LightGBM] [Info] Number of data points in the train set: 69427, number of used features: 42
[LightGBM] [Info] Start training from score 0.082019


In [65]:

# =========================
# VALIDATION SCORE
# =========================

cat_score = r2_score(
    np.expm1(y_valid),
    np.expm1(cat_valid)
)

lgb_score = r2_score(
    np.expm1(y_valid),
    np.expm1(lgb_valid)
)

print("CatBoost R2:", cat_score)
print("LightGBM R2:", lgb_score)


CatBoost R2: 0.8037477368997086
LightGBM R2: 0.7526492329862028


In [66]:

# =========================
# WEIGHTED ENSEMBLE
# =========================

final_valid = (
    0.7 * cat_valid +
    0.3 * lgb_valid
)

ensemble_score = r2_score(
    np.expm1(y_valid),
    np.expm1(final_valid)
)

print("Ensemble R2:", ensemble_score)


Ensemble R2: 0.7915173098877699


In [67]:

# =========================
# FINAL PREDICTIONS
# =========================

final_test = (
    0.7 * cat_test +
    0.3 * lgb_test
)

final_test = np.expm1(final_test)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_test
})

submission.to_csv('submission8.csv', index=False)

submission.head()


,Index,demand
0,0,0.063507
1,1,0.015262
2,2,0.082520
3,3,0.066957
4,4,0.055319



# Additional High-Impact Experiments

## Try These

### 1. Remove KMeans
Sometimes improves score.

---

### 2. Increase CatBoost Depth
Try:
- 9
- 10

---

### 3. More Target Encodings
Try:
- geohash + weather
- geohash + hour
- geo_5 + hour

---

### 4. Pseudo Labeling
Train on confident test predictions.

---

### 5. Weather Severity Encoding
Add ordinal weather score.

---

### 6. Hyperparameter Search
Use Optuna for:
- depth
- l2_leaf_reg
- learning_rate
- bagging_temperature

---

### 7. Cross Validation Ensemble
Train multiple folds and average predictions.
